# Task 2 — Exploratory Data Analysis (EDA)
## E-Commerce Sales Analysis

**Internship Domain:** Data Analytics  
**Task:** Exploratory Data Analysis

The purpose of this notebook is to understand the dataset first, clean basic data-quality issues, and then look for useful sales patterns using simple Python analysis.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import f_oneway

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (9, 5)


## 1. Load the dataset

In [ ]:
df = pd.read_csv("../dataset/ecommerce_sales.csv")

print("Dataset shape:", df.shape)
df.head()


## 2. Questions for the analysis

Before doing the analysis, I wanted to answer these questions:

1. Which category has the highest sales?
2. Which products generate the most sales?
3. Which cities contribute more sales?
4. How do sales change from month to month?
5. Which payment method is used most often?
6. Are there unusual sales values?
7. Are average sales different between categories?


## 3. Understand the data

In [ ]:
print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nBasic information:")
df.info()


## 4. Check missing values and duplicates

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


## 5. Clean the data

In [ ]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"])

# Fill missing values using simple rules.
df["City"] = df["City"].fillna(df["City"].mode()[0])
df["Discount_Percent"] = df["Discount_Percent"].fillna(
    df["Discount_Percent"].median()
)

# Remove exact duplicate records.
df = df.drop_duplicates().reset_index(drop=True)

# Create a month column for trend analysis.
df["Month"] = df["Order_Date"].dt.to_period("M").astype(str)

print("Shape after cleaning:", df.shape)
print("Remaining missing values:", df.isnull().sum().sum())


## 6. Descriptive statistics

In [ ]:
df.describe()


## 7. Sales by category

In [ ]:
category_sales = (
    df.groupby("Category")["Sales"]
      .sum()
      .sort_values(ascending=False)
)

print(category_sales)

category_sales.plot(kind="bar")
plt.title("Total Sales by Category")
plt.xlabel("Category")
plt.ylabel("Sales")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 8. Top products

In [ ]:
product_sales = (
    df.groupby("Product")["Sales"]
      .sum()
      .sort_values(ascending=False)
)

print(product_sales.head(10))

product_sales.head(10).sort_values().plot(kind="barh")
plt.title("Top 10 Products by Sales")
plt.xlabel("Sales")
plt.ylabel("Product")
plt.tight_layout()
plt.show()


## 9. Sales by city

In [ ]:
city_sales = (
    df.groupby("City")["Sales"]
      .sum()
      .sort_values(ascending=False)
)

print(city_sales)

city_sales.head(10).sort_values().plot(kind="barh")
plt.title("Top Cities by Sales")
plt.xlabel("Sales")
plt.ylabel("City")
plt.tight_layout()
plt.show()


## 10. Monthly sales trend

In [ ]:
monthly_sales = df.groupby("Month")["Sales"].sum()

print(monthly_sales)

monthly_sales.plot(kind="line", marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 11. Payment method analysis

In [ ]:
payment_counts = df["Payment_Method"].value_counts()

print(payment_counts)

payment_counts.plot(kind="bar")
plt.title("Orders by Payment Method")
plt.xlabel("Payment Method")
plt.ylabel("Number of Orders")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


## 12. Outlier detection

I used the IQR method to flag unusually high or low sales values. A flagged value is not automatically an error; it simply needs further checking.


In [ ]:
q1 = df["Sales"].quantile(0.25)
q3 = df["Sales"].quantile(0.75)
iqr = q3 - q1

lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

outliers = df[
    (df["Sales"] < lower_limit) |
    (df["Sales"] > upper_limit)
]

print("Lower limit:", round(lower_limit, 2))
print("Upper limit:", round(upper_limit, 2))
print("Possible outliers:", len(outliers))

df.boxplot(column="Sales")
plt.title("Sales Outlier Check")
plt.show()


## 13. Correlation analysis

In [ ]:
numeric_columns = [
    "Quantity",
    "Unit_Price",
    "Discount_Percent",
    "Sales"
]

correlation = df[numeric_columns].corr()
correlation


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(correlation.values, aspect="auto")

ax.set_xticks(range(len(numeric_columns)))
ax.set_yticks(range(len(numeric_columns)))
ax.set_xticklabels(numeric_columns, rotation=30, ha="right")
ax.set_yticklabels(numeric_columns)

for i in range(len(numeric_columns)):
    for j in range(len(numeric_columns)):
        ax.text(
            j, i,
            f"{correlation.iloc[i, j]:.2f}",
            ha="center",
            va="center"
        )

ax.set_title("Correlation Heatmap")
fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()


## 14. Hypothesis testing

**H0:** Average sales are the same across the categories.  
**H1:** At least one category has a different average sales value.

One-way ANOVA is used for this comparison. The 5% significance level is used as the decision rule.


In [ ]:
groups = [
    group["Sales"].values
    for _, group in df.groupby("Category")
]

f_statistic, p_value = f_oneway(*groups)

print("F-statistic:", round(f_statistic, 4))
print("p-value:", round(p_value, 6))

if p_value < 0.05:
    print("Decision: Reject H0 at the 5% significance level.")
else:
    print("Decision: Fail to reject H0 at the 5% significance level.")


## 15. Final observations

From this analysis, I was able to:

- understand the structure of the dataset;
- identify and clean missing values;
- remove duplicate records;
- compare sales across categories, products and cities;
- study the monthly sales pattern;
- check payment-method usage;
- identify possible outliers;
- examine correlations between numerical variables;
- perform a basic hypothesis test using ANOVA.

The results shown above are based on the included demo dataset.


## 16. Conclusion

EDA helped me understand how raw sales records can be converted into useful information. The next step for a larger project could be a dashboard, customer segmentation or sales forecasting.
